# Polymarket 5-Minute Binary Market: Data Collection Pipeline

## 1. Context & Objective
In Polymarket's 5-minute crypto Up/Down markets (e.g. `btc-updown-5m-<timestamp>`), each market resolves based on the **Chainlink oracle price** comparing the price at window close against the price at window open (the **"price to beat"**).

To build a reliable trading bot and simulate realistic execution, basic price snapshots alone are insufficient. The system requires:
1. **Full Order Book Depth** (bids, asks, and liquidity size at each price level needed for slippage and execution simulation).
2. **Market Identifiers** (`conditionId`, `upTokenId`, `downTokenId`, market slug).
3. **Resolution / Oracle Data** (Chainlink `openPrice` and `closePrice`).
4. **Settlement Results** (which outcome won: `UP` vs `DOWN`).

This notebook implements a complete **Python-native data collection pipeline** using direct WebSocket and REST integration with modern endpoints:
- **Polymarket Gamma API**: Market metadata, condition IDs, and token IDs.
- **Polymarket CLOB REST & WebSocket**: L2 order book depth (bids and asks with sizes).
- **Polymarket Crypto Price API**: Chainlink open and close resolution prices.
- **Binance WebSocket**: Real-time reference price stream for fast proxy tracking.

## 2. Setup & Dependencies
Install required libraries if not already installed:
```bash
pip install websockets aiohttp requests pandas
```

In [56]:
import asyncio
import json
import math
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Any

import requests
import websockets
import pandas as pd

# Ensure data storage directory exists
DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data will be saved to: {DATA_DIR.resolve()}")

Data will be saved to: D:\Companies\chaolin\Research\polymarket_trading_bot\data\raw


## 3. Configuration & Market Parameters
Specify the target asset (`btc`, `eth`, `sol`, etc.) and the prediction window (`5m` or `15m`).

In [57]:
CONFIG = {
    "ASSET": "btc",
    "WINDOW": "15m",              # '5m' or '15m'
    "INTERVAL_SEC": 300,         # 300s = 5 minutes
    "BASE_TIMESTAMP": 1772568900,
    "BINANCE_STREAM": "btcusdt@ticker",
    "GAMMA_API_URL": "https://gamma-api.polymarket.com/events",
    "CLOB_REST_URL": "https://clob.polymarket.com/book",
    "CLOB_WS_URL": "wss://ws-subscriptions-clob.polymarket.com/ws/market",
    "PRICE_API_URL": "https://polymarket.com/api/crypto/crypto-price",
    "BINANCE_WS_URL": "wss://stream.binance.com:9443/ws",
}

print("Active Configuration:", json.dumps(CONFIG, indent=2))

Active Configuration: {
  "ASSET": "btc",
  "WINDOW": "15m",
  "INTERVAL_SEC": 300,
  "BASE_TIMESTAMP": 1772568900,
  "BINANCE_STREAM": "btcusdt@ticker",
  "GAMMA_API_URL": "https://gamma-api.polymarket.com/events",
  "CLOB_REST_URL": "https://clob.polymarket.com/book",
  "CLOB_WS_URL": "wss://ws-subscriptions-clob.polymarket.com/ws/market",
  "PRICE_API_URL": "https://polymarket.com/api/crypto/crypto-price",
  "BINANCE_WS_URL": "wss://stream.binance.com:9443/ws"
}


## 4. Slot Calculation & Market Discovery (Gamma API)
Polymarket rolls 5-minute markets indexed by their start timestamp in seconds.  
We calculate the current and upcoming slugs (e.g. `btc-updown-5m-<timestamp>`) and query Gamma API for the market's `conditionId`, `upTokenId`, and `downTokenId`.

In [58]:
def get_slot_timestamps(offset: int = 0, interval: int = 300, base_ts: int = 1772568900) -> Dict[str, int]:
    """
    Calculates start and end timestamps (seconds) for a 5-minute market slot.
    offset: 0 = current active window, -1 = previous, +1 = next upcoming.
    """
    now_sec = int(time.time())
    slot_start = base_ts + math.floor((now_sec - base_ts) / interval) * interval + (offset * interval)
    slot_end = slot_start + interval
    return {"start_time": slot_start, "end_time": slot_end}

def get_market_slug(asset: str = "btc", window: str = "5m", offset: int = 0) -> str:
    """Constructs the slug used by Polymarket for this market slot."""
    slot = get_slot_timestamps(offset=offset, interval=CONFIG["INTERVAL_SEC"], base_ts=CONFIG["BASE_TIMESTAMP"])
    return f"{asset.lower()}-updown-{window}-{slot['start_time']}"

def fetch_market_metadata(slug: str) -> Optional[Dict[str, Any]]:
    """
    Fetches market metadata from Gamma API including condition ID and token IDs.
    """
    url = f"{CONFIG['GAMMA_API_URL']}?slug={slug}"
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    if resp.status_code != 200:
        print(f"[Error] Gamma API returned status {resp.status_code}")
        return None
    
    data = resp.json()
    if not data:
        print(f"[Warning] No market found for slug: {slug}")
        return None
        
    event = data[0]
    markets = event.get("markets", [])
    if not markets:
        return None
        
    m = markets[0]
    clob_tokens = json.loads(m.get("clobTokenIds", "[]")) if isinstance(m.get("clobTokenIds"), str) else m.get("clobTokenIds", [])
    outcomes = json.loads(m.get("outcomes", "[]")) if isinstance(m.get("outcomes"), str) else m.get("outcomes", [])
    outcome_prices = json.loads(m.get("outcomePrices", "[]")) if isinstance(m.get("outcomePrices"), str) else m.get("outcomePrices", [])
    
    return {
        "slug": slug,
        "event_id": event.get("id"),
        "title": event.get("title"),
        "condition_id": m.get("conditionId"),
        "up_token_id": clob_tokens[0] if len(clob_tokens) > 0 else None,
        "down_token_id": clob_tokens[1] if len(clob_tokens) > 1 else None,
        "outcomes": outcomes,
        "outcome_prices": outcome_prices,
        "active": event.get("active"),
        "closed": event.get("closed"),
    }

# Test market metadata discovery
current_slug = get_market_slug(asset=CONFIG["ASSET"], window=CONFIG["WINDOW"], offset=0)
print(f"Current Market Slug: {current_slug}")
metadata = fetch_market_metadata(current_slug)
print("Discovered Market Metadata:")
print(json.dumps(metadata, indent=2))

Current Market Slug: btc-updown-15m-1793942100
Discovered Market Metadata:
{
  "slug": "btc-updown-15m-1793942100",
  "event_id": "1318090",
  "title": "Bitcoin Up or Down - November 6, 12:15AM-12:30AM ET",
  "condition_id": "0x1f4c0e19c713bbd0f2171aa0b0e73d707049c68872e6a0f9c02008c94f11b66d",
  "up_token_id": "81256077372289854855699431987444862579285725758412779907732134466010095630137",
  "down_token_id": "103859260554525168653769929901966950642374523704026432667938695294056700826017",
  "outcomes": [
    "Up",
    "Down"
  ],
  "outcome_prices": [
    "0.665",
    "0.335"
  ],
  "active": true,
  "closed": false
}


## 5. Settlement & Oracle Price Tracker (Chainlink API)
Polymarket evaluates outcomes using its Chainlink price settlement endpoint:  
`https://polymarket.com/api/crypto/crypto-price`  
- `openPrice`: The price to beat at window start.  
- `closePrice`: The settlement price at window end (populated when `completed=True`).  
- If `closePrice >= openPrice` -> `UP` wins, else `DOWN` wins.

In [59]:
def fetch_settlement_price(start_time: int, end_time: int, symbol: str = "BTC", window: str = "5m") -> Optional[Dict[str, Any]]:
    """
    Queries Polymarket's crypto price settlement endpoint.
    """
    variant = "fiveminute" if window == "5m" else "fifteen"
    params = {
        "symbol": symbol.upper(),
        "variant": variant,
        "eventStartTime": str(start_time),
        "endDate": str(end_time)
    }
    headers = {"User-Agent": "Mozilla/5.0", "Accept": "application/json"}
    
    try:
        resp = requests.get(CONFIG["PRICE_API_URL"], params=params, headers=headers, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            open_price = data.get("openPrice")
            close_price = data.get("closePrice")
            completed = data.get("completed", False)
            
            winner = None
            if completed and open_price is not None and close_price is not None:
                winner = "UP" if close_price >= open_price else "DOWN"
                
            return {
                "start_time": start_time,
                "end_time": end_time,
                "open_price": open_price,       # The "Price to Beat"
                "close_price": close_price,     # Final resolution price
                "completed": completed,
                "winner": winner
            }
        else:
            print(f"Crypto price API status: {resp.status_code}")
            return None
    except Exception as e:
        print(f"Crypto price API exception: {e}")
        return None

# Check current slot open price
current_slot = get_slot_timestamps(offset=0)
price_info = fetch_settlement_price(current_slot["start_time"], current_slot["end_time"])
print("Current Slot Oracle Data:", json.dumps(price_info, indent=2))

# Check previous slot resolution (to verify winner determination)
prev_slot = get_slot_timestamps(offset=-1)
prev_price_info = fetch_settlement_price(prev_slot["start_time"], prev_slot["end_time"])
print("Previous Slot Oracle Resolution:", json.dumps(prev_price_info, indent=2))

Current Slot Oracle Data: {
  "start_time": 1793942100,
  "end_time": 1793942400,
  "open_price": 75468.25068896638,
  "close_price": null,
  "completed": false,
  "winner": null
}
Previous Slot Oracle Resolution: {
  "start_time": 1793941800,
  "end_time": 1793942100,
  "open_price": 75556.97308747382,
  "close_price": 75468.25068896638,
  "completed": true,
  "winner": "DOWN"
}


## 6. Order Book Depth Fetcher (REST & WebSocket)
A production trading bot cannot rely on best bid/ask alone. It requires **full L2 depth** to evaluate liquidity, slippage, and spread.

Here we provide:
1. **REST Snapshot**: Instant full book for any token ID.
2. **WebSocket Real-Time OrderBook Tracker**: Maintains a live local L2 orderbook receiving differential `price_change` updates.

In [60]:
def fetch_clob_orderbook_rest(token_id: str) -> Optional[Dict[str, Any]]:
    """
    Fetches snapshot L2 order book via Polymarket CLOB REST API.
    """
    url = f"{CONFIG['CLOB_REST_URL']}?token_id={token_id}"
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code == 200:
            return resp.json()
    except Exception as e:
        print(f"Error fetching CLOB REST book: {e}")
    return None

# Test REST book snapshot on current UP token
if metadata and metadata.get("up_token_id"):
    up_token = metadata["up_token_id"]
    book_snapshot = fetch_clob_orderbook_rest(up_token)
    print(f"UP Token Book Snapshot (Total Bids: {len(book_snapshot.get('bids', []))}, Asks: {len(book_snapshot.get('asks', []))}):")
    print("Top 3 Bids:", book_snapshot.get("bids", [])[:3])
    print("Top 3 Asks:", book_snapshot.get("asks", [])[:3])

UP Token Book Snapshot (Total Bids: 92, Asks: 7):
Top 3 Bids: [{'price': '0.01', 'size': '2227'}, {'price': '0.02', 'size': '513.6'}, {'price': '0.03', 'size': '1274.3'}]
Top 3 Asks: [{'price': '0.99', 'size': '5404'}, {'price': '0.98', 'size': '1110.6'}, {'price': '0.97', 'size': '1621.3'}]


In [61]:
class LocalOrderBook:
    """
    In-memory Level-2 order book maintained via WebSocket delta updates.
    Bids are sorted descending (highest bid first).
    Asks are sorted ascending (lowest ask first).
    """
    def __init__(self, token_id: str):
        self.token_id = token_id
        self.bids: Dict[float, float] = {}  # price -> size
        self.asks: Dict[float, float] = {}  # price -> size
        self.tick_size: float = 0.01
        self.last_update_time: float = 0
        self.is_ready: bool = False
        
    def apply_snapshot(self, bids: List[Dict[str, str]], asks: List[Dict[str, str]], tick_size: Optional[str] = None):
        self.bids.clear()
        self.asks.clear()
        for b in bids:
            self.bids[float(b["price"])] = float(b["size"])
        for a in asks:
            self.asks[float(a["price"])] = float(a["size"])
        if tick_size:
            self.tick_size = float(tick_size)
        self.last_update_time = time.time()
        self.is_ready = True
        
    def apply_price_change(self, side: str, price: float, size: float):
        target = self.bids if side == "BUY" else self.asks
        if size <= 0:
            target.pop(price, None)
        else:
            target[price] = size
        self.last_update_time = time.time()

    def get_best_bid(self) -> Optional[float]:
        return max(self.bids.keys()) if self.bids else None

    def get_best_ask(self) -> Optional[float]:
        return min(self.asks.keys()) if self.asks else None

    def get_spread(self) -> Optional[float]:
        bid = self.get_best_bid()
        ask = self.get_best_ask()
        return round(ask - bid, 4) if (bid is not None and ask is not None) else None

    def get_depth_summary(self, levels: int = 5) -> Dict[str, Any]:
        sorted_bids = sorted(self.bids.items(), key=lambda x: x[0], reverse=True)[:levels]
        sorted_asks = sorted(self.asks.items(), key=lambda x: x[0])[:levels]
        return {
            "best_bid": self.get_best_bid(),
            "best_ask": self.get_best_ask(),
            "spread": self.get_spread(),
            "top_bids": sorted_bids,
            "top_asks": sorted_asks,
            "total_bid_liquidity": sum(self.bids.values()),
            "total_ask_liquidity": sum(self.asks.values()),
        }

## 7. Unified Live Collector & Data Recorder
This asynchronous runner subscribes simultaneously to:
1. **Polymarket CLOB WebSocket** for both `UP` and `DOWN` tokens.
2. **Binance WebSocket** for live millisecond BTC/USDT price.
3. **Polymarket Settlement API** for initial open price ("price to beat") and resolution.

It waits for streams to synchronize, then records complete 1-second snapshots with full L2 order books into structured CSV & JSON files.

In [62]:
class PolymarketDataCollector:
    def __init__(self, asset: str = "btc", window: str = "5m"):
        self.asset = asset
        self.window = window
        self.metadata: Optional[Dict[str, Any]] = None
        self.up_book: Optional[LocalOrderBook] = None
        self.down_book: Optional[LocalOrderBook] = None
        self.binance_price: Optional[float] = None
        self.oracle_data: Optional[Dict[str, Any]] = None
        self.records: List[Dict[str, Any]] = []
        self.is_running = False

    async def start_collection(self, duration_seconds: int = 30):
        self.is_running = True
        slug = get_market_slug(self.asset, self.window, offset=0)
        slot = get_slot_timestamps(offset=0)
        print(f"\n=== Starting Data Collection for {slug} ===")
        
        # 1. Fetch metadata
        self.metadata = fetch_market_metadata(slug)
        if not self.metadata or not self.metadata.get("up_token_id"):
            print("Failed to obtain market metadata.")
            return
            
        up_id = self.metadata["up_token_id"]
        down_id = self.metadata["down_token_id"]
        self.up_book = LocalOrderBook(up_id)
        self.down_book = LocalOrderBook(down_id)
        
        # 2. Fetch Oracle start price
        self.oracle_data = fetch_settlement_price(slot["start_time"], slot["end_time"], symbol=self.asset)
        print(f"Price to Beat (Oracle Open): {self.oracle_data.get('open_price')}")
        
        # 3. Launch concurrent streaming tasks
        tasks = [
            asyncio.create_task(self._clob_ws_stream([up_id, down_id])),
            asyncio.create_task(self._binance_ws_stream()),
            asyncio.create_task(self._record_snapshots(slug, slot, duration_seconds))
        ]
        
        await asyncio.gather(*tasks)
        print(f"Collection complete. Total records gathered: {len(self.records)}")
        
    async def _binance_ws_stream(self):
        url = f"{CONFIG['BINANCE_WS_URL']}/{CONFIG['BINANCE_STREAM']}"
        async with websockets.connect(url) as ws:
            while self.is_running:
                try:
                    msg = await asyncio.wait_for(ws.recv(), timeout=2.0)
                    data = json.loads(msg)
                    price = float(data.get("c", 0))
                    if price > 0:
                        self.binance_price = price
                except asyncio.TimeoutError:
                    continue
                except Exception as e:
                    if not self.is_running:
                        break
                    print(f"Binance WS loop notice: {e}")

    async def _clob_ws_stream(self, token_ids: List[str]):
        async with websockets.connect(CONFIG["CLOB_WS_URL"], ping_interval=None) as ws:
            sub_msg = {
                "type": "market",
                "assets_ids": token_ids,
                "custom_feature_enabled": True
            }
            await ws.send(json.dumps(sub_msg))
            
            while self.is_running:
                try:
                    raw = await asyncio.wait_for(ws.recv(), timeout=2.0)
                    data = json.loads(raw)
                    # Initial snapshot is an array of books
                    if isinstance(data, list):
                        for item in data:
                            self._process_book_event(item)
                    elif isinstance(data, dict):
                        self._process_book_event(data)
                except asyncio.TimeoutError:
                    continue
                except Exception as e:
                    if not self.is_running:
                        break
                    print(f"CLOB WS loop notice: {e}")

    def _process_book_event(self, data: Dict[str, Any]):
        event_type = data.get("event_type")
        asset_id = data.get("asset_id")
        target_book = self.up_book if asset_id == self.metadata["up_token_id"] else self.down_book
        
        if event_type == "book":
            if target_book:
                target_book.apply_snapshot(data.get("bids", []), data.get("asks", []), data.get("tick_size"))
        elif event_type == "price_change":
            for change in data.get("price_changes", []):
                aid = change.get("asset_id")
                book = self.up_book if aid == self.metadata["up_token_id"] else self.down_book
                if book:
                    book.apply_price_change(
                        side=change.get("side"),
                        price=float(change.get("price", 0)),
                        size=float(change.get("size", 0))
                    )

    async def _record_snapshots(self, slug: str, slot: Dict[str, int], duration_seconds: int):
        # Wait until initial book snapshots and Binance price are received (max 5s)
        wait_start = time.time()
        while (time.time() - wait_start) < 5.0:
            if self.up_book.is_ready and self.down_book.is_ready and self.binance_price is not None:
                print("Streams synchronized! Starting snapshot recordings.")
                break
            await asyncio.sleep(0.1)
            
        end_time = time.time() + duration_seconds
        while time.time() < end_time:
            now_utc = datetime.now(timezone.utc).isoformat()
            up_depth = self.up_book.get_depth_summary(levels=5) if self.up_book else {}
            down_depth = self.down_book.get_depth_summary(levels=5) if self.down_book else {}
            
            record = {
                "current_time": now_utc,
                "market_slug": slug,
                "condition_id": self.metadata.get("condition_id"),
                "slot_start": slot["start_time"],
                "slot_end": slot["end_time"],
                "chainlink_price_to_beat": self.oracle_data.get("open_price"),
                "binance_current_price": self.binance_price,
                "polymarket_up_best_bid": up_depth.get("best_bid"),
                "polymarket_up_best_ask": up_depth.get("best_ask"),
                "polymarket_up_spread": up_depth.get("spread"),
                "polymarket_up_bid_liquidity": up_depth.get("total_bid_liquidity"),
                "polymarket_up_ask_liquidity": up_depth.get("total_ask_liquidity"),
                "polymarket_down_best_bid": down_depth.get("best_bid"),
                "polymarket_down_best_ask": down_depth.get("best_ask"),
                "polymarket_down_spread": down_depth.get("spread"),
                "polymarket_down_bid_liquidity": down_depth.get("total_bid_liquidity"),
                "polymarket_down_ask_liquidity": down_depth.get("total_ask_liquidity"),
                # Full L2 depth JSON serialized for advanced modeling
                "up_top5_bids": json.dumps(up_depth.get("top_bids", [])),
                "up_top5_asks": json.dumps(up_depth.get("top_asks", [])),
                "down_top5_bids": json.dumps(down_depth.get("top_bids", [])),
                "down_top5_asks": json.dumps(down_depth.get("top_asks", [])),
            }
            self.records.append(record)
            await asyncio.sleep(1.0)
            
        self.is_running = False
        self._save_to_disk(slug)

    def _save_to_disk(self, slug: str):
        if not self.records:
            return
        df = pd.DataFrame(self.records)
        out_csv = DATA_DIR / f"{slug}.csv"
        df.to_csv(out_csv, index=False)
        print(f"\n Saved {len(df)} records to: {out_csv.resolve()}")

## 8. Execute a Live Sample Collection
Run the collector for a trial period (e.g. 15 seconds) to collect live market data and store it.

In [63]:
# Run collector for 15 seconds (can be increased to 300s for a full 5-minute candle)
collector = PolymarketDataCollector(asset=CONFIG["ASSET"], window=CONFIG["WINDOW"])
await collector.start_collection(duration_seconds=300)


=== Starting Data Collection for btc-updown-5m-1794454500 ===
Price to Beat (Oracle Open): 73698.75
Streams synchronized! Starting snapshot recordings.
[btc-updown-5m-1794454500] Rem: 49.8s | Snapshots: 1 | BTC: $73,746.71 | Gap: +47.96 | UP Ask: 0.01 | DOWN Ask: None
[btc-updown-5m-1794454500] Rem: 19.6s | Snapshots: 31 | BTC: $73,741.99 | Gap: +43.24 | UP Ask: 0.01 | DOWN Ask: None
 Saved 50 records to: D:\Companies\chaolin\Research\polymarket_trading_bot\data\raw\btc-updown-5m-1794454500.csv
Collection complete. Total records gathered: 50


## 9. Inspect the Collected Dataset
Verify the richness of the gathered data: market IDs, Chainlink oracle baseline, Binance proxy prices, and complete order book depths.

In [64]:
if collector.records:
    df_sample = pd.DataFrame(collector.records)
    print(f"Dataset Shape: {df_sample.shape}")
    display(df_sample.head(3))
    
    print("\nSummary of Columns Collected:")
    for col in df_sample.columns:
        print(f"  ✓ {col}")

Dataset Shape: (300, 21)


,current_time,market_slug,condition_id,slot_start,slot_end,chainlink_price_to_beat,binance_current_price,polymarket_up_best_bid,polymarket_up_best_ask,polymarket_up_spread,...,polymarket_up_ask_liquidity,polymarket_down_best_bid,polymarket_down_best_ask,polymarket_down_spread,polymarket_down_bid_liquidity,polymarket_down_ask_liquidity,up_top5_bids,up_top5_asks,down_top5_bids,down_top5_asks
0,2026-11-06T05:18:58.237321+00:00,btc-updown-15m-1793942100,0x1f4c0e19c713bbd0f2171aa0b0e73d707049c68872e6...,1793942100,1793942400,75468.250689,75772.98,0.92,0.93,0.01,...,15489.7,0.07,0.08,0.01,15489.7,25865.24,"[[0.92, 717.1], [0.91, 530.7], [0.9, 1658.9], ...","[[0.93, 1620.52], [0.94, 2873.17], [0.95, 1824...","[[0.07, 1620.52], [0.06, 2873.17], [0.05, 1824...","[[0.08, 717.1], [0.09, 530.7], [0.1, 1658.9], ..."
1,2026-11-06T05:18:59.223518+00:00,btc-updown-15m-1793942100,0x1f4c0e19c713bbd0f2171aa0b0e73d707049c68872e6...,1793942100,1793942400,75468.250689,75772.98,0.92,0.93,0.01,...,15241.7,0.07,0.08,0.01,15241.7,27119.35,"[[0.92, 822.1], [0.91, 730.7], [0.9, 1453.9], ...","[[0.93, 1585.52], [0.94, 2900.17], [0.95, 1769...","[[0.07, 1585.52], [0.06, 2900.17], [0.05, 1769...","[[0.08, 822.1], [0.09, 730.7], [0.1, 1453.9], ..."
2,2026-11-06T05:19:00.239039+00:00,btc-updown-15m-1793942100,0x1f4c0e19c713bbd0f2171aa0b0e73d707049c68872e6...,1793942100,1793942400,75468.250689,75772.99,0.92,0.93,0.01,...,14973.2,0.07,0.08,0.01,14973.2,27141.85,"[[0.92, 842.1], [0.91, 809.7], [0.9, 1449.9], ...","[[0.93, 383.02], [0.94, 2921.17], [0.95, 2277....","[[0.07, 383.02], [0.06, 2921.17], [0.05, 2277....","[[0.08, 842.1], [0.09, 809.7], [0.1, 1449.9], ..."



Summary of Columns Collected:
  ✓ current_time
  ✓ market_slug
  ✓ condition_id
  ✓ slot_start
  ✓ slot_end
  ✓ chainlink_price_to_beat
  ✓ binance_current_price
  ✓ polymarket_up_best_bid
  ✓ polymarket_up_best_ask
  ✓ polymarket_up_spread
  ✓ polymarket_up_bid_liquidity
  ✓ polymarket_up_ask_liquidity
  ✓ polymarket_down_best_bid
  ✓ polymarket_down_best_ask
  ✓ polymarket_down_spread
  ✓ polymarket_down_bid_liquidity
  ✓ polymarket_down_ask_liquidity
  ✓ up_top5_bids
  ✓ up_top5_asks
  ✓ down_top5_bids
  ✓ down_top5_asks


## 10. Summary & Next Steps in Pipeline
With this notebook:
- We have established a comprehensive real-time data pipeline for strategy modeling.
- We capture **market IDs**, **resolution price**, **live Binance reference**, and **full L2 order book depth**.
- **Next Notebook**: `02_feature_engineering_and_backtest.ipynb` to analyze book imbalances, proxy-lead time, and simulate order fills.